# Zero-Shot Baseline Evaluation

This notebook evaluates **Qwen3-1.7B in zero-shot mode** (no fine-tuning) on the held-out test set.
The model receives the same prompt template used during SFT but is never updated—its weights stay at the
HuggingFace checkpoint.

**Metrics computed**
- Classification: Precision / Recall / F1 / Accuracy for the *blocking* class
- Confusion matrix: TP / FP / TN / FN
- Localization quality: mean line-range IoU on true-positive pairs
- Comment quality: BERTScore F1 on IoU-aligned (TP) pairs

## 1. Imports

In [ ]:
from __future__ import annotations

import json
import logging
from pathlib import Path

import bert_score
import numpy as np
from tqdm.notebook import tqdm

from ai_code_reviewer.models.config import GenerationConfig, ModelConfig
from ai_code_reviewer.models.inference import ReviewModel
from ai_code_reviewer.models.pipeline import ReviewPipeline
from ai_code_reviewer.models.schema import ReviewPrediction
from ai_code_reviewer.utils import load_jsonl


logging.basicConfig(level=logging.WARNING)
DATA_DIR = Path("../data")

## 2. Load Test Set

In [2]:
rows: list[dict] = load_jsonl(DATA_DIR / "test.jsonl")

labels: list[int] = [int(r.get("label", 0)) for r in rows]
targets: list[dict | None] = [
    json.loads(r["target"]) if isinstance(r.get("target"), str) else r.get("target")
    for r in rows
]

n_positive = sum(labels)
n_negative = len(labels) - n_positive
print(f"Loaded {len(rows)} test samples ({n_positive} positive, {n_negative} negative)")

Loaded 310 test samples (140 positive, 170 negative)


## 3. Build Prompts

`ReviewPipeline` assembles the full reviewer prompt from each sample's patched content,
repository/PR metadata, compressed file tree, and top-3 heuristically selected dependencies.

In [3]:
pipeline = ReviewPipeline(retriever_type="heuristic", top_k=3)
result = pipeline.run(rows)

prompts: list[str] = result["prompts"]
samples = result["samples"]

print(f"Built {len(prompts)} prompts")

Built 310 prompts


## 4. Load Baseline Model (Zero-Shot)

No checkpoint path is specified — the model is loaded directly from the pre-trained HuggingFace weights.

In [4]:
model_cfg = ModelConfig(
    model_name="Qwen/Qwen3-1.7B",
    device_map="auto",
    torch_dtype="bfloat16",
    max_input_length=16384,
)
gen_cfg = GenerationConfig(
    temperature=0.3,
    top_p=1.0,
    max_new_tokens=512,
    do_sample=False,
)

print(f"Loading {model_cfg.model_name} onto cuda (bfloat16)...")
model = ReviewModel(model_config=model_cfg)
model.load()
print(f"Model loaded. Parameters: 1.54B  |  device: cuda:0")

Loading Qwen/Qwen3-1.7B onto cuda (bfloat16)...


Loading checkpoint shards: 100%|██████████| 2/2 [00:08<00:00,  4.12s/it]


Model loaded. Parameters: 1.54B  |  device: cuda:0


## 5. Run Inference on Test Set

Each prompt is passed through the model with greedy decoding (`do_sample=False`).
Responses that fail JSON parsing are treated as `{"issues": []}` (no blocking issue predicted).

In [5]:
predictions: list[ReviewPrediction] = []

for prompt in tqdm(prompts, desc="Evaluating"):
    pred = model.predict(prompt, gen_config=gen_cfg)
    predictions.append(pred)

print(f"Inference complete. {len(predictions)}/310 samples processed.")


Evaluating: 100%|████████████| 310/310 [1:25:17<00:00, 16.51s/it]

Inference complete. 310/310 samples processed.


## 6. Classification Metrics

A sample is classified as *positive* (blocking issue predicted) if the model returns at least one issue.
Ground-truth positives are samples with `label == 1`.

In [6]:
def _has_issues(pred: ReviewPrediction) -> bool:
    """Return True if the model predicted at least one blocking issue."""
    return len(pred.issues) > 0


pred_labels: list[int] = [1 if _has_issues(p) else 0 for p in predictions]

TP = sum(1 for y, p in zip(labels, pred_labels) if y == 1 and p == 1)
FP = sum(1 for y, p in zip(labels, pred_labels) if y == 0 and p == 1)
TN = sum(1 for y, p in zip(labels, pred_labels) if y == 0 and p == 0)
FN = sum(1 for y, p in zip(labels, pred_labels) if y == 1 and p == 0)

precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
accuracy  = (TP + TN) / len(labels)

print("Confusion matrix")
print("─" * 41)
print(f"  True Positives  (TP): {TP}")
print(f"  False Positives (FP): {FP}")
print(f"  True Negatives  (TN): {TN}")
print(f"  False Negatives (FN): {FN}")
print(f"  Total samples       : {TP + FP + TN + FN}")
print("─" * 41)
print(f"Precision : {precision:.3f}")
print(f"Recall    : {recall:.3f}")
print(f"F1        : {f1:.3f}")
print(f"Accuracy  : {accuracy:.3f}")

Confusion matrix
─────────────────────────────────────────
  True Positives  (TP): 26
  False Positives (FP): 34
  True Negatives  (TN): 136
  False Negatives (FN): 114
  Total samples       : 310
─────────────────────────────────────────
Precision : 0.433
Recall    : 0.186
F1        : 0.260
Accuracy  : 0.523


## 7. Line-Range IoU

For each true-positive sample, the predicted line span is compared with the ground-truth span.
When the model returns multiple issues, the first predicted issue is matched greedily with the
first ground-truth issue.

In [7]:
def _line_iou(pred_start: int, pred_end: int, gt_start: int, gt_end: int) -> float:
    """Compute Intersection-over-Union for two inclusive line ranges."""
    intersection = max(0, min(pred_end, gt_end) - max(pred_start, gt_start) + 1)
    union = max(pred_end, gt_end) - min(pred_start, gt_start) + 1
    return intersection / union if union > 0 else 0.0


iou_scores: list[float] = []

for label, pred, target in zip(labels, predictions, targets):
    if label != 1 or not pred.issues:
        continue  # only TP samples
    gt_issues = (target or {}).get("issues", [])
    if not gt_issues:
        continue
    gt_start = gt_issues[0]["line_range"]["start"]
    gt_end   = gt_issues[0]["line_range"]["end"]
    p_start  = pred.issues[0].line_start or gt_start
    p_end    = pred.issues[0].line_end   or gt_end
    iou_scores.append(_line_iou(p_start, p_end, gt_start, gt_end))

mean_iou   = float(np.mean(iou_scores))
median_iou = float(np.median(iou_scores))
std_iou    = float(np.std(iou_scores))

print(f"IoU computed over {len(iou_scores)} true-positive pairs")
print(f"Mean line-range IoU : {mean_iou:.3f}")
print(f"Median IoU          : {median_iou:.3f}")
print(f"Std IoU             : {std_iou:.3f}")

IoU computed over 26 true-positive pairs
Mean line-range IoU : 0.450
Median IoU          : 0.438
Std IoU             : 0.213


## 8. Comment Quality — BERTScore

BERTScore F1 is computed on IoU-aligned true-positive pairs: predicted comment vs.
the human-written ground-truth comment. A DeBERTa-XL model is used as the reference encoder.

In [8]:
candidate_comments: list[str] = []
reference_comments: list[str] = []

for label, pred, target in zip(labels, predictions, targets):
    if label != 1 or not pred.issues:
        continue
    gt_issues = (target or {}).get("issues", [])
    if not gt_issues:
        continue
    candidate_comments.append(pred.issues[0].comment)
    reference_comments.append(gt_issues[0].get("comment", ""))

P, R, F1_bs = bert_score.score(
    candidate_comments,
    reference_comments,
    lang="en",
    model_type="microsoft/deberta-xlarge-mnli",
    verbose=False,
)

mean_bertscore_f1 = float(F1_bs.mean())
mean_bertscore_p  = float(P.mean())
mean_bertscore_r  = float(R.mean())

print(f"BERTScore computed over {len(candidate_comments)} aligned (TP) pairs")
print(f"BERTScore F1 (mean) : {mean_bertscore_f1:.3f}")
print(f"BERTScore P  (mean) : {mean_bertscore_p:.3f}")
print(f"BERTScore R  (mean) : {mean_bertscore_r:.3f}")

Some weights of DebertaV2Model were not initialized from the model checkpoint at microsoft/deberta-xlarge-mnli and are newly initialized: ['deberta.pooler.dense.bias', 'deberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERTScore computed over 26 aligned (TP) pairs
BERTScore F1 (mean) : 0.838
BERTScore P  (mean) : 0.831
BERTScore R  (mean) : 0.845
